In [ ]:
# scripts/2_baseline.py
"""
Baseline pixel-wise using XGBoost.
- For each pixel, extract features across a small window (e.g., 3x3 or 5x5) for DSM + impervious + canopy + NDVI + slope
- Train to predict height = DSM - lidar_dtm
- Evaluate MAE/RMSE global and per-class using landcover/impervious mask
"""
import os, numpy as np
from glob import glob
import rasterio
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib
import tqdm

DATA_PROC = "../data/processed"
# paths (example patterns)
dsm_files = glob(os.path.join(DATA_PROC, "*_DSM.tif"))
dtm_files = glob(os.path.join(DATA_PROC, "*_lidar_dtm.tif"))  # targets
# naive pairing by base name - adapt to your naming
paired = []
for d in dsm_files:
    base = os.path.splitext(os.path.basename(d))[0].replace("_DSM","")
    dtm = os.path.join(DATA_PROC, base + "_lidar_dtm.tif")
    imp = os.path.join(DATA_PROC, base + "_impervious.tif")
    canopy = os.path.join(DATA_PROC, base + "_canopy.tif")
    ndvi = os.path.join(DATA_PROC, base + "_NDVI.tif")
    slope = os.path.join(DATA_PROC, base + "_slope.tif")
    if os.path.exists(dtm):
        paired.append((d, dtm, imp, canopy, ndvi, slope))

print(f"Found {len(paired)} paired tiles for baseline")

def extract_features_window(arr, x, y, w=1):
    # arr: HxW np array; take window around (x,y), flatten including center
    h, ww = arr.shape
    xs = slice(max(0,y-w), min(h, y+w+1))
    ys = slice(max(0,x-w), min(ww, x+w+1))
    win = arr[xs, ys].flatten()
    return win

X_all = []
y_all = []

for (dsm, dtm, imp, canopy, ndvi, slope) in paired:
    with rasterio.open(dsm) as r_d, rasterio.open(dtm) as r_t:
        dsm_arr = r_d.read(1).astype('float32')
        dtm_arr = r_t.read(1).astype('float32')
    # optional layers
    def read_if(path): 
        return rasterio.open(path).read(1).astype('float32') if path and os.path.exists(path) else None
    imp_arr = read_if(imp)
    canopy_arr = read_if(canopy)
    ndvi_arr = read_if(ndvi)
    slope_arr = read_if(slope)

    H, W = dsm_arr.shape
    # sample subset of pixels (avoid full image for memory)
    rand_idx = np.random.choice(H*W, size=min(50000, H*W//10), replace=False)
    for idx in rand_idx:
        ypix = idx // W
        xpix = idx % W
        feats = []
        # DSM window 3x3
        feats += list(extract_features_window(dsm_arr, ypix, xpix, w=1))
        # impervious 1x1
        if imp_arr is not None:
            feats.append(imp_arr[ypix, xpix])
        else:
            feats.append(0.0)
        if canopy_arr is not None:
            feats.append(canopy_arr[ypix, xpix])
        else:
            feats.append(0.0)
        if ndvi_arr is not None:
            feats.append(ndvi_arr[ypix, xpix])
        else:
            feats.append(0.0)
        if slope_arr is not None:
            feats.append(slope_arr[ypix, xpix])
        else:
            feats.append(0.0)
        height = dsm_arr[ypix, xpix] - dtm_arr[ypix, xpix]
        X_all.append(feats)
        y_all.append(height)

X_all = np.array(X_all, dtype='float32')
y_all = np.array(y_all, dtype='float32')

X_train, X_val, y_train, y_val = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

print("Training XGBoost baseline...")
model = xgb.XGBRegressor(n_estimators=200, max_depth=8, learning_rate=0.1, tree_method='hist', n_jobs=8)
model.fit(X_train, y_train, eval_set=[(X_val,y_val)], early_stopping_rounds=25, verbose=10)

pred = model.predict(X_val)
mae = mean_absolute_error(y_val, pred)
rmse = mean_squared_error(y_val, pred, squared=False)
print("Baseline XGBoost MAE:", mae, "RMSE:", rmse)

joblib.dump(model, "../outputs/xgb_baseline.joblib")
print("Saved baseline model")
